# Scheduling & Monitoring the TPC E-Commerce Pipeline in Production

This document outlines how to operationalize the **TPC_Ecommerce_Pipeline** (bronze → silver → gold → data-quality) for reliable production execution.

---

## 1. Schedule the Pipeline as a Lakeflow Job

The YAML bundle you defined already creates a multi-task job. To run it on a cadence in production:

1. **Deploy the bundle** to the target workspace using `databricks bundle deploy` (Declarative Automation Bundles).
2. **Add a trigger** — either in the bundle YAML under `schedules:` or via the Jobs UI:
   - *Cron example*: `0 6 * * *` runs every day at 06:00 UTC.
   - *File arrival*: trigger when new source files land in the bronze storage path.
   - *Continuous*: for near-real-time pipelines (if needed).
3. **Set `parameters`** at the job level so `catalog_name` is passed to every notebook task automatically .
4. **Choose compute**: attach a job cluster or use serverless compute to avoid idle costs. Serverless is recommended for intermittent pipelines.

---

## 2. Alerting & Notifications

| Event | Recipient | Channel |
|---|---|
| Task failure | `adityakamodiya@cyntexa.com` | Email  |
| Pipeline success (data-quality task) | `adityakodiya@cyntexa.com` | Email  |
| Health rule breach (run > 300s) | Configured recipients | Email + Job UI |

**Best practices:**
- Add **Slack/Teams webhook** notifications in addition to email for faster visibility.
- Add **on-call escalation** policies so critical failures are not missed during off-hours.
- Consider **alert destinations** (Databricks alerting) for custom SLA breaches.

---

## 3. Retry & Failure Strategy

Your YAML already includes sensible retry logic:

| Task | Max Retries | Retry Interval | Timeout |
|---|---|---|---|
| Bronze | 1 | 0s | 360s |
| Silver | 1 | 0s | — |
| Gold | 0 | 900s | — |
| Data Quality | 1 | 60s | 360s |

- **Bronze failures** (ingestion) are retried once — transient cloud/IO errors are handled.
- **Gold** has no retries but a 15-minute retry interval is set if a retry is added later.
- **Data Quality** is the last gate — its failure means the pipeline should stop and alert.
- **Health rules** flag long-running tasks before they hit the hard timeout.

---

## 4. Monitoring Pipeline Runs

### A. Job Run History (UI + System Tables)
- Navigate to **Jobs → TPC_Ecommerce_Pipeline** in the Databricks UI to view run history, durations, and task-level status.
- Query **`system.lakeflow.job_runs`** for programmatic monitoring:
  ```sql
  SELECT run_id, task_key, state, start_time, end_time, 
         DATEDIFF(second, start_time, end_time) AS duration_sec
  FROM system.lakeflow.job_runs
  WHERE job_name = 'TPC_Ecommerce_Pipeline'
  ORDER BY start_time DESC
  LIMIT 20;
  ```

### B. Data Quality Monitoring
- The **data-quality task** (Task 5) writes a reconciliation report to `dev.ops.data_quality_reconciliation`. Monitor this table for PASS/FAIL trends:
  ```sql
  SELECT run_ts, check_name, severity, status, detail
  FROM dev.ops.data_quality_reconciliation
  ORDER BY run_ts DESC;
  ```
- Set up a **Databricks SQL Alert** on this table to notify when `status = 'FAIL'` and `severity = 'CRITICAL'`.

### C. Compute & Cost Monitoring
- Use **`system.billing.usage`** to track daily DBU consumption per job.
- Use **`system.compute.clusters`** to identify idle time or over-provisioned clusters.

---

## 5. Production Readiness Checklist

- [ ] Bundle deployed via CI/CD (GitLab/GitHub Actions → `databricks bundle deploy`)
- [ ] Schedule trigger configured (cron / file arrival)
- [ ] Job-level `catalog_name` parameter set to the production catalog
- [ ] Email + Slack/Teams notifications configured for all critical tasks
- [ ] Health rules with appropriate duration thresholds per task
- [ ] SQL Alert on `ops.data_quality_reconciliation` for CRITICAL FAILs
- [ ] Compute sized correctly (or serverless enabled) to avoid OOM and timeouts
- [ ] Retry policy reviewed for each task (bronze and silver: 1 retry; gold: consider adding)
- [ ] Access control: only pipeline service principal can write to silver/gold schemas
- [ ] Dashboard created to visualize pipeline SLA, run duration, and DQ pass rate over time

---

## 6. CI/CD Promotion (dev → stage → prod)

1. Maintain the bundle YAML in a Git repo.
2. Use **environments** in the bundle to target `dev`, `stage`, and `prod` workspaces.
3. Each environment sets its own `catalog_name` default (dev / stage / prod) and compute policy.
4. PR review → deploy to dev → smoke test → promote to stage → final approval → deploy to prod.

---

*This ensures the pipeline runs reliably on schedule, failures are caught early, and data quality is continuously monitored across environments.*